# Actividad 2: Proceso KDD para Retención de Clientes en Telecomunicaciones

**Objetivo:** Aplicar el proceso KDD completo para preparar los datos y predecir si un cliente de telecomunicaciones se dará de baja (churn).

**Dataset:** Telco Customer Churn de IBM — se carga directamente desde el repositorio público en GitHub.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('Librerías importadas correctamente.')

**Explicación:** Se importan las librerías necesarias. `pandas` y `numpy` para manipulación de datos, `matplotlib` y `seaborn` para gráficos, y `StandardScaler` de `sklearn` para la normalización de variables numéricas.

---
## PASO 1: Selección y Comprensión de Datos (Data Selection)

Cargamos el dataset y entendemos su estructura, las variables disponibles y la variable objetivo.

In [ ]:
url_telco = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url_telco)

print(f'Forma del dataset: {df.shape}')
print(f'\nPrimeras 5 filas:')
df.head()

**Explicación:** Se carga el dataset directamente desde el repositorio público de IBM en GitHub usando `pd.read_csv()`. No es necesario descargar nada. El dataset contiene información de clientes de telecomunicaciones como su perfil, servicios contratados, cuenta y si se dieron de baja (churn).

In [ ]:
df.info()

**Explicación:** `df.info()` muestra el tipo de dato de cada columna, cuántos valores no nulos tiene y el uso de memoria. Esto permite identificar columnas con tipos incorrectos (como `TotalCharges` que debería ser numérica) y posibles valores faltantes.

In [ ]:
df.describe()

**Explicación:** `df.describe()` muestra estadísticas descriptivas de las variables numéricas: media, desviación estándar, mínimo, máximo y cuartiles. Esto da una primera visión de la distribución de los datos.

In [ ]:
print('=== VARIABLE OBJETIVO ===')
print(f"Variable: 'Churn'")
print(f"Valores: {df['Churn'].value_counts().to_dict()}")
print(f"\nProporción de churn:")
print(df['Churn'].value_counts(normalize=True).round(4))

print('\n=== COLUMNAS DEL DATASET ===')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2d}. {col:25s} | Tipo: {df[col].dtype}')

**Explicación:** Se identifica la variable objetivo `Churn` (Yes/No) y se muestra su distribución. También se listan todas las columnas con su tipo de dato. La variable objetivo indica si el cliente se dio de baja del servicio, que es lo que queremos predecir.

### Identificación de variables relevantes

| Categoría | Variables |
|---|---|
| **Demográficas** | `gender`, `SeniorCitizen`, `Partner`, `Dependents` |
| **Servicio** | `tenure`, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| **Cuenta** | `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges` |
| **Objetivo** | `Churn` (Yes/No) |

**Explicación:** Las variables se organizan en categorías para entender mejor el dataset. Las variables demográficas describen al perfil del cliente, las de servicio detallan qué contrató, las de cuenta muestran su actividad financiera, y `Churn` es lo que queremos predecir.

---
## PASO 2: Limpieza de Datos (Data Preprocessing)

Manejo de valores faltantes, datos inconsistentes y duplicados.

In [ ]:
print('=== VALORES NULOS POR COLUMNA ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else 'No hay valores nulos.')

print('\n=== REVISAR ESPACIOS EN BLANCO EN TotalCharges ===')
espacios_blancos = (df['TotalCharges'] == ' ').sum()
print(f'Filas con espacios en blanco en TotalCharges: {espacios_blancos}')

**Explicación:** Se revisan valores nulos con `isnull()` y también se busca espacios en blanco en `TotalCharges`. Esta columna tiene un problema conocido: algunos registros tienen espacios en blanco (' ') en lugar de valores numéricos, lo que impide su conversión automática.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('TotalCharges después de la conversión:')
print(df['TotalCharges'].describe())
print(f'\nValores nulos en TotalCharges después de coerción: {df["TotalCharges"].isnull().sum()}')

**Explicación:** Se convierte `TotalCharges` de tipo object (texto) a numérico usando `pd.to_numeric()`. El parámetro `errors='coerce'` convierte los valores que no se pueden convertir (como espacios en blanco) en NaN. Después de la conversión, se identifican cuántos valores quedaron como nulos.

In [ ]:
filas_antes = df.shape[0]
df = df.dropna(subset=['TotalCharges'])
filas_despues = df.shape[0]

print(f'Filas eliminadas por TotalCharges nulo: {filas_antes - filas_despues}')
print(f'Filas restantes: {filas_despues}')

**Explicación:** Se eliminan las filas donde `TotalCharges` quedó como NaN después de la conversión. Son aproximadamente 11 filas que tenían espacios en blanco, y se eliminan porque son un porcentaje muy bajo del dataset total.

In [ ]:
duplicados = df.duplicated().sum()
print(f'Filas duplicadas encontradas: {duplicados}')

if duplicados > 0:
    df = df.drop_duplicates()
    print(f'Dataset después de eliminar duplicados: {df.shape[0]} filas')
else:
    print('No hay duplicados. No se eliminó nada.')

**Explicación:** Se verifican filas duplicadas usando `df.duplicated()`. Los duplicados no aportan información nueva y pueden sesgar el análisis, por lo que se eliminan si existen.

In [ ]:
print('=== RESUMEN DE LIMPIEZA ===')
print(f'Forma final del dataset limpio: {df.shape}')
print(f'Valores nulos restantes: {df.isnull().sum().sum()}')
print(f'Duplicados restantes: {df.duplicated().sum()}')
print(f'TotalCharges tipo: {df["TotalCharges"].dtype}')

**Explicación:** Se muestra un resumen del dataset después de la limpieza: forma final, si quedan valores nulos o duplicados, y el tipo de dato de `TotalCharges` (ahora es float64). Esto confirma que los datos están listos para la transformación.

---
## PASO 3: Transformación de Datos (Data Transformation)

Preparación de variables para algoritmos de Machine Learning.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('=== VARIABLE OBJETIVO CONVERTIDA ===')
print(f'Distribución de Churn:')
print(df['Churn'].value_counts())
print(f'\nProporción:')
print(df['Churn'].value_counts(normalize=True).round(4))

**Explicación:** Se convierte la variable objetivo `Churn` de texto ('Yes'/'No') a numérico (1/0) usando `.map()`. Los algoritmos de machine learning requieren variables numéricas, y esta conversión es necesaria para poder entrenar modelos predictivos.

In [ ]:
columnas_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f'Variables categóricas a codificar: {columnas_categoricas}')

df_modelo = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True)

print(f'\nForma antes de get_dummies: {df.shape}')
print(f'Forma después de get_dummies: {df_modelo.shape}')
print(f'\nNuevas columnas generadas:')
nuevas = [c for c in df_modelo.columns if c not in df.columns]
for col in nuevas[:15]:
    print(f'  - {col}')
if len(nuevas) > 15:
    print(f'  ... y {len(nuevas) - 15} más.')

**Explicación:** Se identifican automáticamente las columnas categóricas (tipo object) y se convierten en variables dummy usando `pd.get_dummies()`. Cada categoría se transforma en una columna binaria (0 o 1). `drop_first=True` elimina la primera categoría de cada variable para evitar multicolinealidad.

In [ ]:
columnas_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(f'Variables numéricas a escalar: {columnas_numericas}')

print('\nValores ANTES del escalado:')
print(df_modelo[columnas_numericas].describe().round(2))

scaler = StandardScaler()
df_modelo[columnas_numericas] = scaler.fit_transform(df_modelo[columnas_numericas])

print('\nValores DESPUÉS del escalado (media ≈ 0, std ≈ 1):')
print(df_modelo[columnas_numericas].describe().round(4))

**Explicación:** Se aplica `StandardScaler` a las variables numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`) para que tengan media 0 y desviación estándar 1. Esto es importante porque algoritmos como SVM, KNN y redes neuronales son sensibles a la escala. Sin normalización, `TotalCharges` (valores de 0-8000) dominaría sobre `tenure` (valores de 0-72).

In [ ]:
print('=== DATASET FINAL PARA MODELADO ===')
print(f'Forma: {df_modelo.shape}')
print(f'\nPrimeras 5 filas (columnas seleccionadas):')
df_modelo.head()

**Explicación:** Se muestra el dataset final listo para modelado. Todas las variables son numéricas, no hay valores faltantes, las variables numéricas están normalizadas y las categóricas están codificadas como variables dummy.

---
## PASO 4: Resumen y Conclusión

El proceso KDD ha finalizado exitosamente. Los datos están listos para ser utilizados en modelos de Machine Learning.

### 1. Selección y comprensión
- Dataset: Telco Customer Churn de IBM (GitHub).
- ~7,043 registros de clientes de telecomunicaciones.
- Variable objetivo: `Churn` (Yes/No → 1/0).

### 2. Limpieza de datos
- `TotalCharges` convertida de object a float (manejo de espacios en blanco).
- ~11 filas eliminadas por TotalCharges nulo.
- Duplicados verificados y eliminados si los hubiera.
- Dataset limpio listo para transformación.

### 3. Transformación de datos
- `Churn` convertido a int (1/0).
- Variables categóricas codificadas con `get_dummies()`.
- Variables numéricas escaladas con `StandardScaler`.
- Dataset final listo para modelado.

### 4. Datos listos para modelado
El dataset preprocesado puede utilizarse para entrenar modelos de:
- Regresión Logística
- Random Forest
- XGBoost
- Redes Neuronales